# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Loaded {len(df):,} rows. Base declining rate: {df['is_declining_label'].mean():.3f}")

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth reviewing first if it's **stale** (hasn't been
updated in 180+ days) **and** still **visible** (getting at least 500 impressions in the last
90 days) — a page nobody sees doesn't need a refresh yet, no matter how old it is. I add extra
reason codes so a reviewer can see *why* a page scored high, not just that it did:

- `stale_visible_page` — old update, but still getting real traffic
- `declining_with_demand` — trend is down, but people are still searching for it
- `low_ctr_visible_page` — good position (top 20) but a surprisingly low click rate
- `general_refresh_review` — none of the above, but it still made the ranked list

In [ ]:
def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

def suggested_action(reason_str):
    reasons = set(reason_str.split("|"))
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

print("Rule functions defined: reason_codes(), suggested_action()")

## 2. Build the ranked queue (writes the CSV)

Score = `stale x visible x impressions_90d` — deliberately simple and readable: it's zero
unless a page is BOTH stale AND visible, and among qualifying pages it ranks by how much
exposure they have. I rank every page, attach reason codes and a suggested action, then write
the queue to `work/outputs/baseline_action_score.csv` and check Precision@20 / Precision@50
against the base rate.

In [ ]:
os.makedirs("work/outputs", exist_ok=True)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].apply(suggested_action)

ranked = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out_cols = [
    "rank", "content_id", "client_id", "baseline_score", "reason_codes", "suggested_action",
    "is_declining_label", "impressions_90d", "sessions_90d", "avg_position", "ctr",
    "days_since_last_update", "word_count", "trend_direction",
]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
print(f"\nBase rate (random guessing): {base_rate:.3f}")
for k in (20, 50):
    p = precision_at_k(ranked["baseline_score"], ranked["is_declining_label"], k)
    print(f"Precision@{k}: {p:.3f}   ({round(p*k)} of top {k} actually declining)")

## 3. Top-20 review

For each of the top 20 pages: the action, the reason code, a confidence note, and what would
make the pick wrong. I print the table first, then read it row by row below.

In [ ]:
review_cols = ["rank", "content_id", "impressions_90d", "days_since_last_update",
               "avg_position", "ctr", "reason_codes", "suggested_action", "trend_direction"]
top20 = ranked.head(20)[review_cols]
print(top20.to_string(index=False))

**Reading the top 20:** most rows carry the `stale_visible_page` and `declining_with_demand`
reason codes together — high impressions, an update well over 180 days old, and a downward
trend. Confidence is high on these because all three signals agree with each other.

A few rows only carry `stale_visible_page` without `declining_with_demand` (trend is `stable`
or `up`, not `down`) — I'm less confident on those: the page is old and popular, but nothing
says it's actually getting worse. **What would make a top-20 pick wrong:** a page that looks
stale but is intentionally evergreen (e.g. a reference page that never needs updating), or a
page whose traffic drop is seasonal rather than a genuine content problem — the baseline rule
has no way to tell those apart, which is exactly the kind of case a human reviewer should catch.

## 4. Weak picks + leakage check

I look for weak picks (vague `general_refresh_review` reason) inside the top 20, and I
double-check that the score and reason codes never used the label itself, a future window, or
a FlyRank product flag as an input.

In [ ]:
weak_top20 = ranked.head(20)[ranked.head(20)["reason_codes"] == "general_refresh_review"]
print(f"Weak (vague-reason) picks in top 20: {len(weak_top20)}")

not_declining_top20 = ranked.head(20)[ranked.head(20)["is_declining_label"] == 0]
print(f"\nTop-20 rows that are NOT actually declining ({len(not_declining_top20)} of 20):")
print(not_declining_top20[["rank", "content_id", "impressions_90d", "days_since_last_update", "trend_direction"]].to_string(index=False))

print("\n--- Leakage check ---")
print("Score inputs: days_since_last_update, impressions_90d  -> both observable BEFORE any decision, safe.")
print("Reason codes use: days_since_last_update, impressions_90d, trend_direction, avg_position, ctr")
print("  -> trend_direction is used only to EXPLAIN a reason code, never as a weight inside baseline_score itself.")
print("No FlyRank product flags (health_score, priority_score, action_type, needs_ctr_fix) exist in this dataset,")
print("so there is nothing from that category to accidentally leak in.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.